In [ ]:
import arcpy
import csv

fc = r"C:\Users\Krzysztof\Documents\ArcGIS\Projects\Substations3\Substations3.gdb\Census_2020_Block__Intersect_building_join"
csv_path = r"C:\Users\Krzysztof\Desktop\Outage_status.csv"

subs_field = "CONCATENATE_title"
pop_field = "POPL_TOTAL"

csv_name_field = "Name"
csv_out_field = "Outage"

outaged_list_field = "OutagedSubs"
n_total_field = "N_total"
n_out_field = "N_out"
status_field = "ServeStat"
served_pop_field = "ServedPop"
reduced_pop_field = "ReducedPop"
outaged_pop_field = "OutagedPop"

delimiter = "|"

# Building load fields already in your layer
building_load_fields = [
    "LD_CLINIC",
    "LD_HOSP",
    "LD_SCHOOL",
    "LD_FIRE",
    "LD_OPO",
    "LD_POLICE",
    "LD_SOCIAL",
    "LD_COURT",
    "LD_ENT",
    "LD_GOV",
    "LD_POST",
    "LD_RETAIL",
    "LD_FOOD",
    "LD_EV",
    "LD_CHURCH",
]

# Output fields to be created/updated
served_bldg_total_field = "BLDG_SRV_KW"
reduced_bldg_total_field = "BLDG_RED_KW"
outaged_bldg_total_field = "BLDG_OUT_KW"

# --------------------------------------------------
# Helper: add missing fields
# --------------------------------------------------
existing_fields = {f.name for f in arcpy.ListFields(fc)}

def add_double_field(field_name):
    global existing_fields
    if field_name not in existing_fields:
        arcpy.management.AddField(fc, field_name, "DOUBLE")
        existing_fields.add(field_name)

def add_long_field(field_name):
    global existing_fields
    if field_name not in existing_fields:
        arcpy.management.AddField(fc, field_name, "LONG")
        existing_fields.add(field_name)

def add_text_field(field_name, length=1000):
    global existing_fields
    if field_name not in existing_fields:
        arcpy.management.AddField(fc, field_name, "TEXT", field_length=length)
        existing_fields.add(field_name)

# Existing outage/pop fields, in case they do not exist yet
add_text_field(outaged_list_field, 1000)
add_long_field(n_total_field)
add_long_field(n_out_field)
add_text_field(status_field, 20)
add_double_field(served_pop_field)
add_double_field(reduced_pop_field)
add_double_field(outaged_pop_field)

# Building total fields
add_double_field(served_bldg_total_field)
add_double_field(reduced_bldg_total_field)
add_double_field(outaged_bldg_total_field)

# Per-category outage fields
served_cat_fields = []
reduced_cat_fields = []
outaged_cat_fields = []

for fld in building_load_fields:
    suffix = fld.replace("LD_", "")

    srv_fld = f"SRV_{suffix}"
    red_fld = f"RED_{suffix}"
    out_fld = f"OUT_{suffix}"

    add_double_field(srv_fld)
    add_double_field(red_fld)
    add_double_field(out_fld)

    served_cat_fields.append(srv_fld)
    reduced_cat_fields.append(red_fld)
    outaged_cat_fields.append(out_fld)

# --------------------------------------------------
# Load outage CSV
# --------------------------------------------------
outage_dict = {}

with open(csv_path, "r", newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = (row.get(csv_name_field) or "").strip()
        raw_out = (row.get(csv_out_field) or "").strip()

        if not name:
            continue

        if raw_out in ("0", "1"):
            outage_dict[name] = int(raw_out)

# --------------------------------------------------
# Cursor fields
# --------------------------------------------------
fields = (
    [
        subs_field,
        pop_field,
        outaged_list_field,
        n_total_field,
        n_out_field,
        status_field,
        served_pop_field,
        reduced_pop_field,
        outaged_pop_field,
    ]
    + building_load_fields
    + [served_bldg_total_field, reduced_bldg_total_field, outaged_bldg_total_field]
    + served_cat_fields
    + reduced_cat_fields
    + outaged_cat_fields
)

fully_served_total = 0.0
reduced_total = 0.0
fully_outaged_total = 0.0
no_data_total = 0.0

served_building_total = 0.0
reduced_building_total = 0.0
outaged_building_total = 0.0

with arcpy.da.UpdateCursor(fc, fields) as cur:
    for row in cur:
        subs_raw = row[0] or ""
        pop = float(row[1] or 0)

        subs_all = [s.strip() for s in subs_raw.split(delimiter) if s.strip()]
        subs_known = [s for s in subs_all if s in outage_dict]

        n_total = len(subs_known)
        outaged_subs = [s for s in subs_known if outage_dict[s] == 1]
        n_out = len(outaged_subs)

        outaged_list = delimiter.join(outaged_subs)

        # Building category loads
        bldg_start = 9
        bldg_loads = [float(row[bldg_start + i] or 0) for i in range(len(building_load_fields))]
        bldg_total = sum(bldg_loads)

        if n_total == 0:
            status = "NO_DATA"
            served_pop = 0
            reduced_pop = 0
            outaged_pop = 0

            service_factor = None
            no_data_total += pop

        elif n_out == 0:
            status = "FULL"
            served_pop = pop
            reduced_pop = 0
            outaged_pop = 0

            service_factor = 1.0
            fully_served_total += pop

        elif n_out < n_total:
            status = "REDUCED"
            served_pop = 0
            reduced_pop = pop
            outaged_pop = 0

            service_factor = 0.6
            reduced_total += pop

        else:
            status = "OUTAGE"
            served_pop = 0
            reduced_pop = 0
            outaged_pop = pop

            service_factor = 0.0
            fully_outaged_total += pop

        if service_factor is None:
            served_cat = [0 for _ in bldg_loads]
            reduced_cat = [0 for _ in bldg_loads]
            outaged_cat = [0 for _ in bldg_loads]
        elif service_factor == 1.0:
            served_cat = bldg_loads
            reduced_cat = [0 for _ in bldg_loads]
            outaged_cat = [0 for _ in bldg_loads]
        elif service_factor == 0.6:
            served_cat = [0 for _ in bldg_loads]
            reduced_cat = bldg_loads
            outaged_cat = [0 for _ in bldg_loads]
        else:
            served_cat = [0 for _ in bldg_loads]
            reduced_cat = [0 for _ in bldg_loads]
            outaged_cat = bldg_loads

        served_bldg = sum(served_cat)
        reduced_bldg = sum(reduced_cat)
        outaged_bldg = sum(outaged_cat)

        served_building_total += served_bldg
        reduced_building_total += reduced_bldg
        outaged_building_total += outaged_bldg

        row[2] = outaged_list[:1000]
        row[3] = n_total
        row[4] = n_out
        row[5] = status
        row[6] = served_pop
        row[7] = reduced_pop
        row[8] = outaged_pop

        total_start = 9 + len(building_load_fields)
        row[total_start] = served_bldg
        row[total_start + 1] = reduced_bldg
        row[total_start + 2] = outaged_bldg

        cat_start = total_start + 3

        for i, val in enumerate(served_cat):
            row[cat_start + i] = val

        red_start = cat_start + len(served_cat)
        for i, val in enumerate(reduced_cat):
            row[red_start + i] = val

        out_start = red_start + len(reduced_cat)
        for i, val in enumerate(outaged_cat):
            row[out_start + i] = val

        cur.updateRow(row)

# Refresh layer display
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

for lyr in m.listLayers():
    if lyr.name == "Census_2020_Block__Intersect_Overlaps":
        lyr.visible = False
        lyr.visible = True
        print("Layer refreshed.")
        break

print("Done.")
print(f"Fully served population: {fully_served_total:,.2f}")
print(f"Reduced-service population: {reduced_total:,.2f}")
print(f"Fully outaged population: {fully_outaged_total:,.2f}")
print(f"No-data population: {no_data_total:,.2f}")

print(f"Fully served building load kW: {served_building_total:,.2f}")
print(f"Reduced-service building load kW: {reduced_building_total:,.2f}")
print(f"Fully outaged building load kW: {outaged_building_total:,.2f}")

